# API и маршруты

## Структура маршрутов

Все маршруты определены в `lib/not_myself_cleaning_web/router.ex`.

## Публичные маршруты

Доступны без аутентификации.

### Главная страница

```
GET /
```

**Описание:** Редирект на `/login` (если не авторизован) или `/requests` (если авторизован)

**Контроллер:** `AuthController.home/2`

### Регистрация

```
GET /register
```

**Описание:** Отображение формы регистрации

**Контроллер:** `AuthController.register_page/2`

**Шаблон:** `auth_html/register.html.heex`

```
POST /register
```

**Описание:** Обработка регистрации нового пользователя

**Контроллер:** `AuthController.register/2`

**Параметры:**
- `login` (string, required) - логин (3-30 символов)
- `password` (string, required) - пароль (минимум 6 символов)
- `full_name` (string, required) - полное имя
- `phone` (string, required) - телефон
- `email` (string, required) - email

**Ответ:**
- 302 Redirect → `/requests` (успех)
- 422 Unprocessable Entity (ошибка валидации)

### Вход

```
GET /login
```

**Описание:** Отображение формы входа

**Контроллер:** `AuthController.login_page/2`

**Шаблон:** `auth_html/login.html.heex`

```
POST /login
```

**Описание:** Аутентификация пользователя

**Контроллер:** `AuthController.login/2`

**Параметры:**
- `login` (string, required) - логин
- `password` (string, required) - пароль

**Ответ:**
- 302 Redirect → `/requests` или `/admin` (успех)
- 401 Unauthorized (неверные учетные данные)

**Примечание:** Hardcoded администратор `adminka/password` редиректит на `/admin`

## Защищенные маршруты (требуется аутентификация)

### Выход

```
POST /logout
```

**Описание:** Выход из системы (удаление сессии)

**Контроллер:** `AuthController.logout/2`

**Ответ:**
- 302 Redirect → `/login`

### Заявки пользователя

```
GET /requests
```

**Описание:** Список заявок текущего пользователя

**Контроллер:** `RequestController.index/2`

**Шаблон:** `request_html/index.html.heex`

**Ответ:** HTML страница со списком заявок

```
GET /requests/new
```

**Описание:** Форма создания новой заявки

**Контроллер:** `RequestController.new/2`

**Шаблон:** `request_html/new.html.heex`

```
POST /requests
```

**Описание:** Создание новой заявки

**Контроллер:** `RequestController.create/2`

**Параметры:**
- `address` (string, required) - адрес уборки
- `contact` (string, required) - контактные данные
- `service` (string, required) - тип услуги
- `date_time` (string, required) - дата и время
- `payment` (string, required) - способ оплаты

**Ответ:**
- 302 Redirect → `/requests` (успех)
- 422 Unprocessable Entity (ошибка валидации)

## Административные маршруты (требуется is_admin = true)

### Админ-панель

```
GET /admin
```

**Описание:** Список всех заявок

**Контроллер:** `AdminController.index/2`

**Шаблон:** `admin_html/index.html.heex`

**Доступ:** Только для администраторов

**Ответ:** HTML страница со всеми заявками

```
POST /admin/status
```

**Описание:** Обновление статуса заявки

**Контроллер:** `AdminController.update_status/2`

**Параметры:**
- `id` (uuid, required) - ID заявки
- `status` (string, required) - новый статус (in_work, done, cancelled)
- `comment` (string, required) - комментарий к изменению статуса

**Ответ:**
- 302 Redirect → `/admin` (успех)
- 422 Unprocessable Entity (ошибка валидации)

**Ограничения:**
- Нельзя изменить статус, если текущий статус `done` или `cancelled`
- Из статуса `new` можно перейти в `in_work` или `cancelled`
- Из статуса `in_work` можно перейти в `done` или `cancelled`

## Middleware и Plugs

### Browser Pipeline

Применяется ко всем маршрутам:

```elixir
pipeline :browser do
  plug :accepts, ["html"]
  plug :fetch_session
  plug :fetch_flash
  plug :protect_from_forgery
  plug :put_secure_browser_headers
  plug :load_current_user
end
```

### load_current_user

Загружает текущего пользователя из сессии и добавляет в `conn.assigns.current_user`.

### require_auth (RequestController)

Проверяет наличие авторизованного пользователя. Редиректит на `/login` если не авторизован.

### require_admin (AdminController)

Проверяет флаг `is_admin`. Редиректит на `/requests` если не администратор.

## CSRF защита

Все POST-запросы требуют CSRF токен:

```html
<input type="hidden" name="_csrf_token" value={get_csrf_token()}>
```

Токен автоматически проверяется middleware `protect_from_forgery`.

## Коды ответов

| Код | Описание                          | Использование                     |
|-----|-----------------------------------|-----------------------------------|
| 200 | OK                                | Успешный GET-запрос               |
| 302 | Found (Redirect)                  | Успешный POST, редирект           |
| 401 | Unauthorized                      | Неверные учетные данные           |
| 422 | Unprocessable Entity              | Ошибка валидации формы            |